# പാഠം 18: ക്രിപ്‌റ്റോഗ്രാഫിക് റിസിപ്റ്റുകളിലൂടെ AI ഏജൻറുകൾ സുരക്ഷിതമാക്കൽ

## അനുഭവപാഠപുസ്തകം

ഈ പാഠപുസ്തകം നാലു അഭ്യാസങ്ങൾ നടത്തുന്നു:

1. ഏജൻറ് ടൂൾ കോളിന് വേണ്ടി ആദ്യ רിസിപ്റ്റ് **സൈൻ ചെയ്‌തു** അത് പരിശോധിക്കുക.
2. റിസിപ്റ്റ് **തട്ടിപ്പിടിക്കുക** കൂടാതെ പരിശോധന പരാജയപ്പെടുന്നതു കാണുക.
3. മൂന്ന് റിസിപ്റ്റ് മുണ്ടായ ഒരു ശൃംഖല **സൃഷ്ടിച്ച്** ശൃംഖലയുടെ നിലയറിഞ്ഞു ഉറപ്പാക്കുക.
4. ഓരോ പ്രവർത്തനവും ഒരു റിസിപ്റ്റ് നൽകുന്ന വിധം **Microsoft ഏജൻ്റ് ഫ്രെയിംവർക്കിന്റെ ടൂൾ കോളിനെ പൊതിയുക**.

എല്ലാ ക്രിപ്‌റ്റോഗ്രാഫിക് പ്രിമിറ്റീവുകളും നന്നായി സംരക്ഷിക്കപ്പെട്ട ഗ്രന്ഥാലയങ്ങളിൽ നിന്ന് ഇറക്കുമതി ചെയ്യപ്പെട്ടതാണ് (`pynacl` എഡ്25519-നുള്ളത്, `jcs` RFC 8785 കാനോണിക്കൽ JSON-നുള്ളത്, `hashlib` പൈത്തൺ സ്റ്റാൻഡേഡ് ലൈബ്രറിയിൽ നിന്ന് SHA-256-നുള്ളത്). റിസിപ്റ്റ് തത്ത്വശാസ്ത്രം ശുദ്ധമായ പൈത്തണിലാണ്, നിങ്ങൾക്കു വായിക്കുകയും മാറ്റുകയും ചെയ്യാം.

സെല്ലുകൾ ക്രമേണ പ്രവർത്തിപ്പിക്കുക. ഓരോ വിഭാഗവും നേരിയതിനും സ്വയം സമ്പൂർണവുമാണ്.


## സെറ്റ് അപ്

ഈ രണ്ട് പേരിട്ട Dependencies ഇൻസ്റ്റാൽ ചെയ്യുക. ഇരുവർക്കും അനുമതിപ്രദമായ ലൈസൻസുകൾ ഉണ്ട് (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## സഹായക ഉപകരണങ്ങൾ

ഈ രണ്ട് സഹായികൾ പാഡിംഗ് ഇല്ലാതെ ബേസ്64url എൻകോഡിംഗ് ചെയ്യാനും, യാദൃശ്ചിക വസ്തുക്കളുടെ SHA-256 ഹാഷിംഗ് നടത്താനും ആണ്. അവ ബാക്കി നോട്ട്ബുക്ക് സ്വീകരണ തന്ത്രത്തിനേ സുരക്ഷിതമായി കേന്ദ്രീകരിക്കാൻ സഹായിക്കുന്നു.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## വിഭാഗം 1: നിങ്ങളുടെ ആദ്യപ്പെട്ടി ഒപ്പിടുക

**Contoso Travel** എന്നതിന്റെ ഏജന്റ് സിഡ്‌നി മുതൽ ലോസ് ആഞ്ചലസിലേക്കുള്ള ഫ്ലൈറ്റുകൾ ഒരു ഉപഭോക്താവിന് അന്വേഷിച്ച് എടുത്തു എന്ന് ചിന്തിക്കുക. ഈ ടൂൾ കോളിന്റെ ഗേറ്റ്വേ പ്രസ്താവന രേഖപ്പെടുത്തണം যাতে ഭാവിയിൽ ഒരു ഓഡിറ്റർ അത് വേർതിരിച്ച പിന്ഡ് ഇഷ്യുയർ കീയുടെ കീഴിൽ പരിശോധിക്കാനാകും.

### ഘട്ടം 1.1: ഒപ്പിടുന്ന ഒരു കീ സൃഷ്‌ടിക്കുക

പ്രൊഡക്ഷൻ ൽ, ഏജന്റിന്റെ ഒപ്പിടുന്ന കീ ഹാർഡ്‌വെയർ സെക്യൂരിറ്റി മോഡ്യൂൾ (HSM), അജ്യൂർ കീ വാൾട്ട്, അല്ലെങ്കിൽ സമാനമായ ഒരു സംരക്ഷിത സ്റ്റോറിൽ വയ്ക്കപ്പെട്ടു നടക്കും. ഈ പാഠത്തിനായി നാം മെമ്മറിയിൽ പുതിയത് ഒരു കീ സൃഷ്‌ടിക്കുന്നു.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### ഘടകം 1.2: റസീറ്റ് പേലോഡ് നിർമ്മിക്കുക

പേലോഡിൽ റസീറ്റ് സ്ഥിരീകരിക്കേണ്ട എല്ലാ വിവരങ്ങളും അടങ്ങിയിരിക്കുന്നു: ആരാണ് പ്രവർത്തിച്ചത്, ഏത് ഉപകരണം, ഏത്_ARGUMENTS_, എന്ത് തിരിച്ചുവന്നു, ഏത് നയം പ്രകാരം, എപ്പോൾ._ARGUMENTS_ മിന്നൽ തിരിച്ച് ഉൾപ്പെടുത്തുന്നതിന് പകരം ഞങ്ങൾ അവയുടെ ഹാഷ് എടുത്തുനിൽക്കുന്നു, അതുകൊണ്ട് റസീറ്റ് ലുക്എസ് ഉള്ള ഉള്ളടക്കം ചോര്‍ക്കാതെ വരുന്നു.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### ഘട്ടം 1.3: സ്വീകരണം ഒപ്പിടുക,ശേഖരിക്കുക

മൂന്ന് ഘട്ടങ്ങള്‍:

1. JCS ഉപയോഗിച്ച് പെയ്‌ലോഡ് canonicalize ചെയ്യുക അങ്ങനെ രണ്ടു നടപ്പാക്കലുകളും സമാനമായ യുക്തിപരമായ സ്വീകരണം സൃഷ്‌ടിയ്ക്കുമ്പോള്‍ ബൈറ്റ്-ഐഡന്റിക്കല്‍ ബൈറ്റുകള്‍ ഉണ്ടാവും.
2. Ed25519 സ്വകാര്യ കീ ഉപയോഗിച്ച് canonical ബൈറ്റുകളെ നേരിട്ട് ഒപ്പിടുക. PureEdDSA സന്ദേശം അകത്തായി ഹാഷ് ചെയ്യുന്നതിനാല്‍ ഒരു അധിക പ്രി-ഹാഷ് പ്രോട്ടോക്കോളിനെ മാറ്റും.

ഒപ്പാണ് പിന്നീട് അന്തിമ സ്വീകരണം സൃഷ്‌ടിയ്ക്കാന്‍ മ исходному peyload-ന് കൂടെ ചേര്‍ക്കുക.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### ഘട്ടം 1.4: സ്വീകരണം പരിശോധിക്കുക

പരിശോധന പ്രക്രിയ തിരിച്ച് നടത്തുകയാണ്. ഞങ്ങൾ ഒപ്പിടലുകൾ നീക്കം ചെയ്യുന്നു, ഔദ്യോഗിക ബൈറ്റ്‌കൾ വീണ്ടും കണക്കുകൂട്ടുന്നു, അതും ഒപ്പിടൽ നിർമാതാവിന്റെ പൊതു കീ ഉപയോഗിച്ച് പരിശോധിക്കുന്നു.

പരിശോധന ഓഫ്‌ലൈൻ ആണ്: വിളിക്കാനോ നെറ്റ്വർക്ക് ആക്‌സസ് വേണ്ട. ഇതിന് സ്വീകരണം കൂടാതെ മറ്റൊന്ന് വേണം: ഒപ്പിടൽ നിർമാതാവ് പൊതു കീ, നേരത്തെ ഓഡിറ്റർ വിശ്വസിക്കുന്ന ഒരു ചാനലിലൂടെ ലഭിച്ചതായിരിക്കണം, ഉദാഹരണത്തിന് ഒപ്പിടൽ നിർമാതാവ് പ്രസിദ്ധീകരിക്കുന്ന ഒരു JWK സെറ്റ് അല്ലെങ്കിൽ മുൻകൂട്ടി നിശ്ചയിച്ച ഒരു മൂല്യം. സ്വീകരണത്തിനുള്ളിലുള്ള `public_key` ഫീൽഡ് ആ പങ്ക് വഹിക്കാനാകില്ല, കാരണം സ്വീകരണം എഴുതുന്നത് ആരാണോ അവർക്ക് ആ ഫീൽഡും എഴുതാൻ സാധിക്കും. തട്ടിപ്പുകൾ തടയുന്ന ഭാഗം 2 ൽ കാണാം.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


നിങ്ങൾ കാണേണ്ടത് `Receipt is valid: True`, `Pre-hashed receipt valid: False`, `Malformed receipts rejected: True` എന്നിവയാണ്. പോസിറ്റീവ് കേസ് നേരിട്ടുള്ള JCS ഒപ്പ് വെക്കൽ പാത പ്രവർത്തിക്കുന്നുവെന്ന് തെളിയിക്കുന്നു; നെഗറ്റീവ് കൺട്രോൾ ഒപ്പ്-വ്യാപ്തി നിയമം പ്രൊസ്‌സായി നീണ്ടുപോകാതെ പ്രവർത്തിക്കാവുന്നതാക്കി മാറ്റുന്നു; അവസാനം ഉള്ള വരി തെറ്റായ ഇൻപുട്ട് ഒരു uitzonderണം ഉയർത്താതെ `False` തിരികെ നൽകുന്നതായി കാണിക്കുന്നു.


## വിഭാഗം 2: റസീത് മാറ്റിവെക്കൽ

റസീതുകളുടെ മുഴുവൻ ഉദ്ദേശവും അവ മാറ്റം വരുത്താൻ പറ്റാത്തതായിരിക്കുകയാണ്. അത് നമുക്ക് തെളിയിക്കാം.

ഒരു വെരിഫിക്കേഷൻ പരാജയപ്പെടുന്നതായി കാണാൻ, നമുക്ക് റസീതിന്റെ ഒരു കഥാപാത്രം മാത്രം മാറ്റാൻ പോകുന്നു.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### What just happened?

When we changed `policy_id`, the canonical bytes changed. The signature (which was over the original canonical bytes) no longer matches. Verification correctly returns `False`.

There is no way to change a signed field, meaning anything outside the `signature` object, and still have the receipt verify against the issuer's public key, unless the attacker has the issuer's private key. The `signature` object itself is not signed: `sig` is the signature, and `verify_receipt` ignores `public_key`, so editing that field alone changes nothing. As long as the private key is in a key vault and verifiers use the issuer's independently published key, a change to a signed field is impossible to hide. That condition matters: the next cell shows a modified receipt that verifies perfectly against a different key.

Try it yourself: modify the `tool_name` or `agent_id` or `timestamp` in the cell above and re-run. Every change to a signed field produces an invalid receipt.


### ഓർമ്മയിൽപ്പെടാത്ത തകർച്ച

മുകളിൽ കാണിച്ചിരിക്കുന്ന ആക്രമകൻ പുനഃസംയോജനം ചെയ്തിട്ടില്ല. കൂടുതൽ ശ്രദ്ധവാനായ ഒരാൾ പത്രം മാറ്റിവെക്കും,
അതിൻറെ കീ ഒരു സ്വന്തമായി സൃഷ്ടിച്ച കീയോടെ ഒപ്പുവെച്ച്, ആ കീ ആ വിമാനത്തിൽ ഇടൂ
`public_key` ഫീൽഡ്. ഫലമായി ലഭിക്കുന്നത് ഒത്തിരിക്ക് ഏറ്റവും സത്യസന്ധമായ ഒപ്പ് ഉള്ള ഒരു രാസീത് ആണ്
അതിന്റെ ഉള്ളടക്കത്തിന്.  

ഇത് പ്രവർത്തിപ്പിച്ച് ഈ രണ്ട് പരിശോധികളിൽ ഏത് പിടിക്കുന്നുവെന്ന് നോക്കുക.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


നിങ്ങള്‍ കാണണം `True` പിന്നീട് `False`. ആ മൂന്ന് `assert` വരികള്‍ അത് ഒരു
പരിശോധന ആക്കുന്നു: forgery ആ issuerന്റെ കീക്ക് എപ്പോഴും സാധുവാകുകയാണെങ്കില്‍ സെല്‍ ഒരു പിഴവ് ഉയര്‍ത്തും.


forgery അന്തര്‍ഗതമായി സുസ്ഥിരമാണ്. അതിന്റെ ഒപ്പു അതിന്റെ ഉള്ളടക്കങ്ങളോടും അത് നിലനിര്‍ത്തുന്ന കീയോടും യഥാര്‍ഥമായി പൊരുത്തപ്പെടുന്നു, അതിനാല്‍ കീ റെസീറ്റ്‌ല്‍ നിന്നു വായിക്കുന്ന ഒരാള്‍ അതിനെ സാധുവായി റിപ്പോര്‍ട്ട് ചെയ്യും. റെസീറ്റില്‍ ഒന്നും തെറ്റായതുപോലെയില്ല, കാരണം അതില്‍ ഒന്നും അസംഗതമായതല്ല.




അത് പിടിക്കുന്ന ഏക കാര്യം ആക്രമണകാരിക്ക് ബാധിക്കാനുള്ള വഴി ഇല്ലാത്ത ഒരു കീയെതിരെയുള്ള പരിശോധിക്കലാണ്. 
ഇത് കാരണം `verify_receipt` കീ ഒരു.Argument വാക്കിയായി സ്വീകരിക്കുന്നു: 
`verify_receipt(receipt)` എന്ന ഒപ്പ് അടയാളമുള്ള ഒരു ഫംഗ്ഷൻ സുരക്ഷിതമായി എഴുതാൻ കഴിയില്ല,
കാരണം അത് എത്തിച്ചേരാൻ കഴിയുന്ന ഏക കീ ആക്രമണകാരൻ നൽകി.

അതേ റസീറ്റ് `sample_receipts/04_resigned_receipt.json` ആയി സംരക്ഷിക്കും 
നിങ്ങൾക്ക് നോട്ട്‌ബുക്ക് പുറത്തും അത് പരിശോധിക്കാനുണ്ടെങ്കിൽ.


## വകുപ്പ് 3: റെസിപ്റ്റുകൾ തമ്മിൽ ബന്ധിപ്പിക്കുക

ഒറ്റ ഒരു റെസിപ്റ്റ് ഒരു പ്രവർത്തിയെ സംരക്ഷിക്കുന്നു. უმეტეს ഏജന്റ്മാർ നിരവധി പ്രവർത്തനങ്ങൾ നടത്താറുണ്ട്. മുഴുവൻ പരമ്പരവും മാറ്റം കാണിക്കുന്നതെന്ന് ഉറപ്പാക്കാൻ, ഓരോ റെസിപ്റ്റും മുൻപത്തെ റെസിപ്റ്റിന്റെ ഹാഷ് പുതിയ റെസിപ്റ്റിന്റെ പേലോഡ് ഉൾപ്പെടുത്തുന്നതിലൂടെ മുൻപത്തെതുമായി ബന്ധിപ്പിക്കുന്നു.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

ഒരുക്കിയ പരമ്പരയിലുള്ള റെസിപ്റ്റ് ഒരാളെ നീക്കംചെയ്യേണ്ടോ പുനഃക്രമീകരിക്കേണ്ടോ ചെയ്താൽ അടുത്ത ബന്ധം തകരും. ട്രങ്കേഷനും ഒപ്പം സിഗ്നർ നിയന്ത്രിത അബദ്ധരേഖ പുനഃലേഖനവും കണ്ടെത്തുന്നതിനായി ഇപ്പോഴുള്ള തലPinned current head, ബാഹ്യ ചെക്ക്പോയിന്റ്, അല്ലെങ്കിൽ അംഗീകരിക്കപ്പെട്ട ജനസംഖ്യ കമ്മിറ്റ്‌മെന്റ് ആവശ്യമാണ്.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


ഇപ്പോൾ മധ്യത്തിലുളള رسید ് മാറ്റം വരുത്തിയിട്ട് ശൃംഖല തകർത്ത് വീണ്ടും പരിശോധന നടത്തൂ. മാറ്റം വരുത്തിയ رسید് തന്റെ ഒപ്പ് പരിശോധനയിൽ പരാജയപ്പെടുന്നു, കൂടാതെ അടുത്ത رسید് തന്റെ ശൃംഖല-കണ്ണി പരിശോധനയിൽ പരാജയപ്പെടുന്നു (എന്തെന്നാൽ അതിന്റെ `previous_receipt_hash` ഇനി മാറ്റം വന്ന മധ്യ رسیدന്റെ ഹാഷുമായി പൊരുത്തമില്ല).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


റിസീറ്റ് 0 ഇപ്പോഴും പരിശോധിക്കുന്നു (അത് മാറ്റം വരുത്തിയിട്ടില്ല, കൂടാതെ ആശ്രയിക്കേണ്ട മുൻക്കാരനും ഇല്ല). റിസീറ്റ് 1-এর ഒപ്പിടൽ പരിശോധന പരാജയപ്പെടുന്നു കാരണം ഞങ്ങൾ `tool_args_hash` മാറ്റിയിട്ടുണ്ട്. റിസീറ്റ് 2-ന്റെ ചൈൻ-ലിങ്ക് പരിശോധന പരാജയപ്പെടുന്നു കാരണം അതിന്റെ `previous_receipt_hash` യഥാർത്ഥ (ഇപ്പോൾ മാറ്റം വരുത്തിയ) റിസീട്ട് 1 ന് എതിരെ കണക്കാക്കിയതാണ്.

ഒരു ആക്രമകൻ മാറ്റം വരുത്തിയ റിസീട്ട് 1 വീണ്ടും ഒപ്പുവയ്ക്കalsa പോലും, പുതിയ ഒപ്പിടൽ ഇഷ്യൂവറുടെ കീ നൽകാത്തവരായി പരാജയപ്പെടും, അവർക്ക് ഇഷ്യൂവറുടെ സ്വകാര്യ കീ ഇല്ലെങ്കിൽ (തങ്ങളുടെ സ്വന്തം കീ ഉപയോഗിച്ച് ഒപ്പിടൽ പരാജയപ്പെടും, സെക്ഷൻ 2-ൽ കാണിച്ചതുപോലെ), കൂടാതെ റിസീട്ട് 2-ലെ ചൈൻ-ലിങ്ക് വ്യത്യാസം ഇപ്പോഴും തട്ടിപ്പിനെ വെളിപ്പെടുത്തും. മാറ്റം മറയ്ക്കാൻ, ആക്രമകനെ മാറ്റം വരുത്തിയ സ്ഥാനത്തിന്നും തുടർന്നുള്ള ഓരോ റിസീറ്റ്-ഉം വീണ്ടും ഒപ്പിടേണ്ടി വരും, അത് ചെയ്യാൻ ഇഷ്യൂവറിന്റെ സ്വകാര്യ കീ ആവശ്യമാകും.


## വിഭാഗം 4: ഒരു ഏജന്റ് ടൂൾ വിളിക്ക് റെസീപ്പ് ഒപ്പിടലോടെ റാപ്പുചെയ്യൽ

യഥാർത്ഥ വിന്യാസത്തിൽ, എല്ലാ ഏജന്റ് എഴുത്തുകാരും `make_receipt` വിളിക്കാൻ ഓർക്കണമെന്ന് നിങ്ങൾ ആഗ്രഹിക്കുന്നില്ല. ഓരോ ടൂൾ വിളിക്കലിനും റെസീപ്പ് ഒപ്പിടലുള്ളത് സ്വയംപോലെയെന്നിരിക്കണം.

ഇതാ ഏറ്റവും ലളിതമായ മാതൃക: ഏത് വിളിക്കാവുന്ന ടൂൾ ഫംഗ്ഷനുമായി ചേർന്ന് ഒരു റെസീപ്പ് പുറപ്പെടുവിക്കുന്ന പതിപ്പ് തിരികെ നൽകുന്ന ഒരു റാപ്പർ ക്ലാസ്. ഇത് Microsoft Agent Framework (`agent_framework.foundry`) ഉൾപ്പെടെയുള്ള ഏജന്റ് ഫ്രെയിംവർക്ക് ഇത് വഴങ്ങുന്നു.

Microsoft Foundry പദ്ധതി സജ്ജമാക്കാത്ത പക്ഷം, താഴെയുള്ള പ്രാദേശിക മոքു മൂലം മാതൃക കാണിക്കുന്നു.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്കുമായി സംയോജിപ്പിക്കൽ

മേൽ പറയുന്ന `ReceiptedTool` റാപ്പർ ഫ്രെയിംവർക്കിൽ സ്വതന്ത്രമാണ്. മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്കിൽ നിർമ്മിച്ച ഏജന്റിനകത്ത് ഇത് ഉപയോഗിക്കാൻ, റാപ്പ് ചെയ്ത ഫംഗ്ഷനെ ഒരു ടൂളായി രജിസ്റ്റർ ചെയ്യുക. ഒരു രൂപരേഖ (നിങ്ങൾ മ്രോക്ന പകരം യഥാർത്ഥ മൈക്രോസോഫ്റ്റ് ഫൗണ്ട്രി ടൂൾ രജിസ്‌ട്രേഷൻ ഉപയോഗിക്കും):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

ഏജന്റ് ഫ്രെയിംവർക്കിനു റിസിപ്റ്റ് സംബന്ധിച്ചൊന്നും അറിയേണ്ടതില്ല. റിസിപ്റ്റ് സൈൻ ചെയ്യൽ ടൂളിന്റെ ചുറ്റും റാപ്പ് ചെയ്തതായും, ഫ്രെയിംവർക്കിന്റെ ഭാഗമല്ലാതെ ഉണ്ടായിരിക്കുന്നു. ഏജന്റിന്റെ കോഡ് പുനർനിർമ്മിക്കാതെ നിലവിലുള്ള ഏജന്റ് കോഡിലേക്ക് ഇതുപോലൊരു പ്രാവണൻസ് ചേർക്കുന്ന രീതിയാണ് ഇത്.


## പുനര്‍വിശ്ലേഷണംയും വിപുലമായ വെല്ലുവിളിയും

നിങ്ങൾ ചെയ്തിട്ടുണ്ട്:

- Ed25519 കീ ജോഡിയെ സൃഷ്ടിച്ചു.
- ഒരു ഏജന്റ് ടൂൾ വിളിക്കലിനുള്ള രേഖ നിർമ്മിച്ച് ഒപ്പിട്ടു.
- രസീത് ഓഫ്‌ലൈൻ തരത്തിൽ ഉപയോഗിച്ച് വെരിഫൈ ചെയ്തു, അക്കാര്യം ലഭിക്കാതെ വെരിഫിക്കേഷൻ ചെയ്തിരിക്കുന്നു.
- ഒരു രസീത് തട്ടിപ്പ് ചെയ്തു, വെരിഫിക്കേഷൻ പരാജയം കാണിച്ചു.
- മൂന്ന് രസീതുകളുടെ ഹാഷ്-ചെയ്ന് ചെയ്ത സീറീസ് നിർമ്മിച്ചു.
- ചങ്ങലയുടെ നടുക്കിൽ തട്ടിപ്പ് ചെയ്ത് ഒപ്പിടൽ പരാജയം കൂടാതെ ചങ്ങല-കണ്ണി പരാജയം مشاهിച്ചു.
- ഒരു ടൂൾ ഫംഗ്ഷനെ ഓട്ടോമാറ്റിക് റസീത് ഒപ്പിട്ടലോടെ പൊതിഞ്ഞു.

**വിപുലീകരണ വെല്ലുവിളി.** ഒരു `request_id` ഫീൽഡ് (വിതരണ ട്രേസിംഗിനുള്ള UUID) റസീത് സ്കീമയിൽ ചേർക്കുക. `make_receipt` അപ്ഡേറ്റ് ചെയ്ത് അത് ഉൾപ്പെടുത്തുക, പിന്നെ രസീത് മുഴുവൻ അന്തരീക്ഷം പരിശേധിച്ച് പരിശോധിക്കുക. ഒപ്പിടുന്നതിന് ശേഷം ഫീൽഡ് മാറ്റിയ ശേഷം വെരിഫിക്കേഷൻ പരാജയം കാണുക. ഇതിലൂടെ കാനോണിക്കൽ എൻ‌കോഡിങ്ങിന്റെ ഓരോ ബൈട്ട് ഒപ്പിടലിൽ എങ്ങനെ പങ്ക് വഹിക്കുന്നു എന്ന് ഉൾക്കൊള്ളേണ്ടിവരും.

**പ്രധാനം സീമ.** പിന്ഡ് കീയും ചങ്ങലാ സന്ദർഭവും ഉള്ളപ്പോൾ, രസീത് ഒപ്പിടൽ കൃത്യത, ഉള്ളടക്കത്തിന്റെ സമഗ്രത, മുൻവഴിയുള്ള ബന്ധം തെളിയിക്കുന്നു. അവ പ്രവർത്തനം, ഉറവിടസത്യൻ, വിശ്വസനീയമായ സമയം, പൂർണ്ണത, നയ നിരൂപണം, അല്ലെങ്കിൽ ശരിയായതിന്റെ തെളിവ് നൽകുന്നില്ല. രസീത് ഒരു അടിസ്ഥാനം മാത്രമാണ്. ഗവേണൻസ് അത് മുകളിൽ നിർമ്മിക്കുന്ന സംവിധാനം ആണ്.

ആ പരിധി മനസ്സിലാക്കി പാഠം README വീണ്ടും വായിക്കുക. പ്രധാനപ്പെട്ട തെറ്റായ ധാരണ രസീത്ത് ഉള്ളതെന്നു കരുതുന്നത് "നാം നിയന്ത്രിതരാണ്" എന്നാണു. അത് സത്യമായില്ല. രസീത്ത് ഏജന്റ് പെരുമാറ്റം പരിശോധിക്കാൻ സഹായിക്കും. അത് ശരിയാക്കുകയില്ല.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
